# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

ONE ROW =
  One article/page, one date, one country

TIME WINDOW =
  90 days trailing from decision_date
  (e.g., if decision is 2026-03-31, use 2026-01-01 to 2026-03-31)
  
  Why 90 days?
  - Long enough to see ranking trends (30 days = too noisy)
  - Short enough to catch recent decay (180 days = misses urgency)
  - Matches editorial refresh cycle (refresh every 3 months or so)

EXAMPLE ROW:
  page_url: flyrank.com/how-to-email-marketing
  date: 2026-03-15
  country: US
  impressions: 145
  clicks: 12
  position: 4.2
  bounce_rate: 0.38
  
RESULT:
  ~90 rows per article per country in your training data
  (if you pick US only, ~90 rows per article)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# 2. Fields: Feature / Label / Context / Excluded

## FEATURES (data knowable at decision time, no leakage)

- impressions_90d: Total searches that showed this article, last 90 days. Knowable? YES (trailing data).
- clicks_90d: Total clicks from search, last 90 days. Knowable? YES (trailing data).
- avg_position_90d: Average rank position, last 90 days. Knowable? YES (trailing aggregate).
- position_trend: Rank velocity (day 60-90 vs day 1-30). Is it dropping? Knowable? YES (calculated from trailing).
- ctr_actual_90d: Actual clicks / impressions, last 90 days. Knowable? YES (trailing).
- ctr_expected_for_position: Benchmark CTR for that rank position. Knowable? YES (locked benchmark).
- ctr_gap: 1 - (actual_ctr / expected_ctr). CTR we're leaving on table. Knowable? YES (derived).
- bounce_rate_90d: % users who left immediately. Knowable? YES (GA4 data, trailing).
- time_on_page_90d: Average seconds on page. Knowable? YES (GA4 data, trailing).
- content_word_count: Word count of article. Knowable? YES (metadata, static).
- days_since_publish: Article age (publish_date - decision_date). Knowable? YES.
- days_since_last_update: Days since last refresh. Knowable? YES (metadata).

## LABEL (the ranking proxy)

refresh_opportunity_score = (impressions_90d) × (ctr_gap) × (position_trend_velocity) ÷ (content_word_count/1000)

Reasoning: High volume + low CTR + dropping ranks + shorter content = high ROI. Rank all articles 1-N. Top 50 = "refresh this week".

## CONTEXT (metadata, explains rows but not used in model)

- page_url: Which page?
- country: Which country?
- date: Which date?
- target_keyword: What keyword ranking for?
- content_type: Blog / guide / tutorial / etc.?
- title: Current article title?

## EXCLUDED (and WHY)

- is_branded = TRUE → Why: Can't improve ranking by refreshing branded queries. Different strategy.
- days_since_publish < 14 → Why: Too new. Rank volatility normal. Wait 2 weeks.
- impressions_90d < 10 → Why: No volume = no opportunity. Not worth editor time.
- country NOT IN ('US', 'GB', 'CA') → Why: Focusing on English markets first.
- content_type IN ('evergreen', 'legal', 'news') → Why: Different refresh strategies.
- actual_ctr > expected_ctr_95th_percentile → Why: Already optimized. No room for improvement.
- position < 0.5 → Why: Data error or featured snippet. Exclude malformed.
- days_since_last_update < 30 → Why: Last refresh was recent. Give it 30 days to settle. Don't refresh twice in one month.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# First, load the data
import pandas as pd
from datasets import load_dataset

# Load the sample table (small one, for testing)
dataset = load_dataset("FlyRank/internship-warehouse", data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Check: how many rows? What's the shape?
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'FlyRank/internship-warehouse' is a gated dataset on the Hub. You must be authenticated to access it.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.